# SFP / SC Port Pose — YOLOv11-pose Fine-tuning

Fine-tunes `yolo11s-pose` (pretrained on COCO) to detect SFP port 0, SFP port 1, and SC port
and predict 5 keypoints per port.  The keypoints are fed to `cv2.solvePnP` to recover the
6-DoF port pose used by `_gt_approach`.

## Data annotation workflow (do this before running the notebook)

1. Collect **40–60 screenshots per class** from the simulation using
   `my_policy_node/scripts/capture_port_screenshots.sh` (separate script).
   Screenshots come from the centre camera at varied viewpoints — no auto-labelling.

2. Upload images to **[Roboflow](https://roboflow.com)** (free tier is sufficient).
   - Create project → **Keypoint Detection**
   - Classes: `sfp_port0`, `sfp_port1`, `sc_port`
   - Use the **SAM2 smart-polygon** tool to auto-segment the port housing, then
     place 5 keypoints manually in this order:
       - **KP 0** — port centre (centre of the socket opening)
       - **KP 1** — top-left corner of socket face
       - **KP 2** — top-right corner of socket face
       - **KP 3** — bottom-right corner of socket face
       - **KP 4** — bottom-left corner of socket face
   - For SC: KP 0 = centre, KP 1–4 = 4 cardinal points on the circular flange edge

3. In Roboflow: **Generate** → augmentation off (handled in training) → **Export**
   → format **YOLOv8** → download the zip.

4. Upload `port_dataset.zip` to the same directory as this notebook on the cluster,
   then run cell 2 to extract it.

## Keypoint 3D geometry (port-local frame, entrance_link)

| KP | SFP (metres) | SC (metres) |
|----|-------------|-------------|
| 0  | [0, 0, 0] | [0, 0, 0] |
| 1  | [−6, −4, 0] mm | [+1.3, 0, 0] mm |
| 2  | [+6, −4, 0] mm | [0, +1.3, 0] mm |
| 3  | [+6, +4, 0] mm | [−1.3, 0, 0] mm |
| 4  | [−6, +4, 0] mm | [0, −1.3, 0] mm |

## Approx. training time on RTX A5000 (25 GB)

100 epochs × ~120 images → **~3–4 minutes**

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True
)
if r.returncode == 0:
    print(f"GPU: {r.stdout.strip()}")
else:
    print("⚠  No GPU detected — check your cluster job allocation")

In [ ]:
# ── Paths — upload port_dataset.zip next to this notebook before running ──────
import os, zipfile
from pathlib import Path

BASE_PATH  = Path(os.getcwd())
OUTPUT_DIR = BASE_PATH / "yolo_output"
OUTPUT_DIR.mkdir(exist_ok=True)

ZIP_PATH    = BASE_PATH / "port_dataset.zip"   # ← upload your Roboflow export zip here
DATASET_DIR = BASE_PATH / "port_dataset"

if not DATASET_DIR.exists():
    if ZIP_PATH.exists():
        print(f"Extracting {ZIP_PATH} ...")
        with zipfile.ZipFile(ZIP_PATH) as zf:
            zf.extractall(BASE_PATH)
        # Roboflow zips extract to a project-named folder — find and rename it
        candidates = [p for p in BASE_PATH.iterdir()
                      if p.is_dir() and p.name not in ("yolo_output",) and p != DATASET_DIR]
        if candidates:
            newest = max(candidates, key=lambda p: p.stat().st_mtime)
            if newest != DATASET_DIR:
                newest.rename(DATASET_DIR)
        print(f"Dataset extracted to {DATASET_DIR}")
    else:
        print(f"⚠  {ZIP_PATH} not found — upload port_dataset.zip to {BASE_PATH} and re-run")
else:
    print(f"Dataset already at {DATASET_DIR} ✓")

print(f"Output dir : {OUTPUT_DIR}")

In [ ]:
# ── Train / val split ────────────────────────────────────────────────────────
# Handles two input layouts:
#   A) PoseAutoLabeler zip: yolo_labeled/ with sfp_nic/ and sc_port/ subdirs
#   B) Flat zip:            images/ + labels/ in DATASET_DIR directly
import random, shutil, yaml

# ── A: Merge PoseAutoLabeler per-class subdirs into flat images/ + labels/ ───
# If the zip extracted to yolo_labeled/ (per-class layout), merge into DATASET_DIR.
yolo_labeled_dir = BASE_PATH / "yolo_labeled"
if yolo_labeled_dir.exists() and not (DATASET_DIR / "images").exists():
    print(f"Merging PoseAutoLabeler output: {yolo_labeled_dir} → {DATASET_DIR}")
    (DATASET_DIR / "images").mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels").mkdir(parents=True, exist_ok=True)
    n_merged = 0
    for cls_dir in sorted(yolo_labeled_dir.iterdir()):
        if not cls_dir.is_dir() or cls_dir.name.startswith("."):
            continue
        src_img = cls_dir / "images"
        src_lbl = cls_dir / "labels"
        if not src_img.is_dir():
            continue
        for img_path in sorted(src_img.glob("*.png")):
            dst_stem = f"{cls_dir.name}_{img_path.stem}"
            shutil.copy(img_path, DATASET_DIR / "images" / f"{dst_stem}.png")
            lbl_src  = src_lbl / (img_path.stem + ".txt")
            if lbl_src.exists():
                shutil.copy(lbl_src, DATASET_DIR / "labels" / f"{dst_stem}.txt")
            n_merged += 1
    cls_dirs = [d.name for d in yolo_labeled_dir.iterdir() if d.is_dir()]
    print(f"  Merged {n_merged} images from classes: {cls_dirs}")
    # Carry over data.yaml written by PoseAutoLabeler
    src_yaml = yolo_labeled_dir / "data.yaml"
    if src_yaml.exists():
        shutil.copy(src_yaml, DATASET_DIR / "data.yaml")
        print(f"  Copied data.yaml")

# ── Diagnostics ───────────────────────────────────────────────────────────────
print(f"\nDATASET_DIR: {DATASET_DIR}  (exists={DATASET_DIR.exists()})")
if DATASET_DIR.exists():
    for p in sorted(DATASET_DIR.iterdir()):
        n = len(list(p.glob("*"))) if p.is_dir() else ""
        print(f"    {p.name}/  ({n} items)" if p.is_dir() else f"    {p.name}")

# ── B: 80/20 train/val split from flat images/ + labels/ ─────────────────────
img_dir = DATASET_DIR / "images"
lbl_dir = DATASET_DIR / "labels"

if img_dir.exists() and not (DATASET_DIR / "train").exists():
    images = [p for p in sorted(img_dir.glob("*"))
              if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")]
    print(f"\nFlat dataset: {len(images)} images — creating 80/20 train/val split ...")
    random.seed(42)
    random.shuffle(images)
    n_val      = max(1, int(len(images) * 0.2))
    val_imgs   = images[:n_val]
    train_imgs = images[n_val:]
    for split, split_imgs in [("train", train_imgs), ("valid", val_imgs)]:
        (DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
        (DATASET_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
        for img_path in split_imgs:
            shutil.copy(img_path, DATASET_DIR / split / "images" / img_path.name)
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                shutil.copy(lbl_path, DATASET_DIR / split / "labels" / lbl_path.name)
    print(f"  train: {len(train_imgs)}  valid: {len(val_imgs)}")
elif (DATASET_DIR / "train").exists():
    t = len(list((DATASET_DIR / "train" / "images").glob("*")))
    v = len(list((DATASET_DIR / "valid" / "images").glob("*")))
    print(f"\nSplit already exists — train: {t}  valid: {v}")
else:
    print("\n⚠  No images/ or train/ found — check DATASET_DIR")

# ── Build / update data.yaml ──────────────────────────────────────────────────
yaml_path = DATASET_DIR / "data.yaml"
if yaml_path.exists():
    with open(yaml_path) as f:
        data_cfg = yaml.safe_load(f) or {}
else:
    data_cfg = {}

data_cfg["path"]      = str(DATASET_DIR.resolve())
data_cfg["train"]     = "train/images"
data_cfg["val"]       = "valid/images"
data_cfg["kpt_shape"] = [10, 3]   # 10 keypoints × (x, y, visibility)
if "names" not in data_cfg:
    # sfp_nic: both SFP ports on one NIC card (kp0-4=port0, kp5-9=port1)
    # sc_port:  SC fibre port (kp0-4 real, kp5-9 invisible padding)
    data_cfg["names"] = ["sfp_nic", "sc_port"]
    data_cfg["nc"]    = 2

with open(yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"\ndata.yaml → {yaml_path}")
print(yaml.dump(data_cfg, default_flow_style=False))

In [ ]:
# ── Fix label format: add visibility column if Roboflow exported x,y only ────
# Roboflow YOLOv8 pose exports 15 cols (no visibility); YOLO needs 20 (vis=2 per kp).
# This cell is idempotent — safe to re-run.

# First: show what the labels actually look like
sample_lbl = next((DATASET_DIR / 'train' / 'labels').glob('*.txt'), None)
if sample_lbl:
    first_line = sample_lbl.read_text().strip().splitlines()[0]
    ncols = len(first_line.split())
    print(f'Sample label: {sample_lbl.name}')
    print(f'  cols={ncols}: {first_line[:120]}')
    if ncols == 20:
        print('  → already 20 columns (x y vis per keypoint) — no fix needed')
    elif ncols == 15:
        print('  → 15 columns (x y only per keypoint) — will add visibility=2')
    else:
        print(f'  → unexpected column count {ncols} — check your Roboflow export')
else:
    print('⚠  No label files found in train/labels/')

def _fix_labels_dir(lbl_dir):
    fixed = already_ok = skipped = 0
    for lf in Path(lbl_dir).glob('*.txt'):
        lines_in  = lf.read_text().strip().splitlines()
        lines_out = []
        changed   = False
        for line in lines_in:
            if not line.strip(): continue
            vals = line.split()
            n = len(vals)
            if n == 20:
                lines_out.append(line)
            elif n == 15:
                new_vals = vals[:5]
                for i in range(5):
                    base = 5 + i*2
                    new_vals += [vals[base], vals[base+1], '2']
                lines_out.append(' '.join(new_vals))
                changed = True
            else:
                lines_out.append(line)
                skipped += 1
        if changed:
            lf.write_text('\n'.join(lines_out) + '\n')
            fixed += 1
        else:
            already_ok += 1
    return fixed, already_ok, skipped

for split in ('train', 'valid', 'test'):
    lbl_dir = DATASET_DIR / split / 'labels'
    if lbl_dir.exists():
        f, ok, sk = _fix_labels_dir(lbl_dir)
        print(f'  {split}: fixed {f}  already_ok {ok}  skipped {sk}')

# Delete ALL stale YOLO cache files so training re-reads the fixed labels
deleted = []
for cache in DATASET_DIR.rglob('*.cache'):
    cache.unlink()
    deleted.append(cache.name)
if deleted:
    print(f'\nDeleted caches: {deleted}')
else:
    print('\nNo cache files found (clean state)')
print('\n✓ Ready — re-run training cell now')

In [ ]:
# Install ultralytics (provides yolo11)
import subprocess, sys, site

subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "ultralytics", "roboflow", "-q", "--user"])

# Append (not prepend) user site-packages so ultralytics is visible but
# conda packages (matplotlib, etc.) still take priority.
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

from ultralytics import YOLO
import ultralytics
print(f"Ultralytics {ultralytics.__version__}")

import torch
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os, json, yaml, shutil, math, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## Inspect the dataset

Verify the split and data.yaml look correct before training.

```
port_dataset/
  data.yaml
  train/
    images/  *.jpg
    labels/  *.txt   (YOLO pose format)
  valid/
    images/
    labels/
```

In [ ]:
# Verify data.yaml
yaml_path = DATASET_DIR / "data.yaml"
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)
print("data.yaml contents:")
print(yaml.dump(data_cfg, default_flow_style=False))

# Count images
for split in ("train", "valid"):
    img_dir = DATASET_DIR / split / "images"
    if img_dir.exists():
        n = len(list(img_dir.glob("*.*")))
        print(f"  {split}: {n} images")

In [ ]:
# Patch data.yaml: absolute paths + enforce correct kpt_shape and class list
data_cfg["path"] = str(DATASET_DIR.resolve())
for split in ("train", "valid", "test"):
    if split in data_cfg:
        if not Path(data_cfg[split]).is_absolute():
            data_cfg[split] = f"{split}/images"

# 10 keypoints per instance:
#   sfp_nic  → kp0-4: port_0 (center,TL,TR,BR,BL)  kp5-9: port_1 (same layout)
#   sc_port  → kp0-4: port keypoints  kp5-9: invisible (visibility=0)
data_cfg["kpt_shape"] = [10, 3]
data_cfg["names"]     = ["sfp_nic", "sc_port"]
data_cfg["nc"]        = 2

patched_yaml = OUTPUT_DIR / "data.yaml"
with open(patched_yaml, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)
print(f"Patched data.yaml → {patched_yaml}")
print(f"  classes  : {data_cfg['names']}")
print(f"  kpt_shape: {data_cfg['kpt_shape']}")

In [ ]:
# Visualise a few annotated training images
if (DATASET_DIR / "train" / "images").exists():
    train_img_dir = DATASET_DIR / "train" / "images"
    train_lbl_dir = DATASET_DIR / "train" / "labels"
else:
    train_img_dir = DATASET_DIR / "images"
    train_lbl_dir = DATASET_DIR / "labels"
    print("⚠  Using flat images/ dir — run the split cell first")

class_names = data_cfg.get("names", [])

# 10 colours: warm=port0 (kp0-4), cool=port1 (kp5-9)
KP_COLORS_VIS = [
    (50,220,50),  (220,200,0),  (220,140,0),  (220,60,0),   (200,80,60),   # port 0
    (0,180,220),  (120,80,220), (180,80,200), (200,80,180),  (120,200,220), # port 1
]
# Corner order within each port: TL(+1)→TR(+2)→BR(+3)→BL(+4)→TL(+1)
PORT_RECT_REL = [1, 2, 3, 4]
PORT_OFFSETS  = [0, 5]

def parse_yolo_pose_label(txt_path, img_w, img_h, n_kp=10):
    """Returns list of (class_id, bbox_xyxy, kps) where kps=[(u,v,vis)×n_kp]."""
    instances = []
    for line in open(txt_path).read().strip().split("\n"):
        if not line: continue
        vals = list(map(float, line.split()))
        cls  = int(vals[0])
        cx, cy, bw, bh = vals[1]*img_w, vals[2]*img_h, vals[3]*img_w, vals[4]*img_h
        bbox = (cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2)
        kps  = []
        for i in range(n_kp):
            base = 5 + i*3
            kps.append((vals[base]*img_w, vals[base+1]*img_h, vals[base+2])
                       if base+2 < len(vals) else (0., 0., 0.))
        instances.append((cls, bbox, kps))
    return instances

exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
imgs = [p for p in sorted(train_img_dir.glob("*")) if p.suffix.lower() in exts][:8]
print(f"Showing {len(imgs)} images from {train_img_dir}")

if not imgs:
    print("⚠  No images found")
else:
    fig, axes = plt.subplots(2, 4, figsize=(22, 9))
    for ax in axes.flatten():
        ax.axis("off")
    for ax, img_path in zip(axes.flatten(), imgs):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        lbl_path = train_lbl_dir / (img_path.stem + ".txt")
        title_parts = []
        if lbl_path.exists():
            for cls_id, bbox, kps in parse_yolo_pose_label(lbl_path, W, H):
                x1,y1,x2,y2 = [int(v) for v in bbox]
                cv2.rectangle(img, (x1,y1), (x2,y2), (200,200,0), 2)
                name = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
                cv2.putText(img, name, (x1, max(y1-5,10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,0), 1)
                title_parts.append(name)
                # Draw port rectangle outlines
                for port_start in PORT_OFFSETS:
                    pts = [(int(kps[port_start+r][0]), int(kps[port_start+r][1]))
                           for r in PORT_RECT_REL
                           if port_start+r < len(kps) and kps[port_start+r][2] > 0]
                    if len(pts) == 4:
                        col = (50,200,50) if port_start==0 else (50,100,220)
                        for j in range(4):
                            cv2.line(img, pts[j], pts[(j+1)%4], col, 1)
                # Draw keypoints
                for i, (u, v, vis) in enumerate(kps):
                    if vis > 0:
                        c = KP_COLORS_VIS[i % len(KP_COLORS_VIS)]
                        cv2.circle(img, (int(u),int(v)), 5, c, -1)
                        cv2.putText(img, str(i), (int(u)+6,int(v)+5),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.32, c, 1)
        ax.imshow(img); ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(", ".join(title_parts) or "(no label)", fontsize=8)
    plt.suptitle("Training annotations  (green rect=port0, blue rect=port1)", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "annotations_preview.png", dpi=100); plt.show()
    print("⟶ Verify keypoints land precisely on port faces before training")

## Fine-tune YOLOv11s-pose

`yolo11s-pose` is pre-trained on COCO 2017 (keypoints task).  We freeze the backbone for
the first 10 epochs then unfreeze for fine-tuning — this is controlled by the `freeze` arg.

With ~120 curated images, 150 epochs takes ~3–4 min on an RTX A5000.

In [ ]:
# ── Hyperparameters — adjust based on your dataset size ──────────────────────
EPOCHS        = 150    # increase to 200 for < 60 images
IMGSZ         = 640    # YOLO standard; images are resized internally
BATCH         = 16     # RTX A5000 (25 GB) can handle 16 comfortably; reduce to 8 if OOM
LR0           = 0.001  # initial LR
LRF           = 0.01   # final LR = LR0 * LRF
FREEZE_LAYERS = 10     # freeze first N backbone layers during warmup
PATIENCE      = 50     # early stopping patience
NUM_WORKERS   = 4      # dataloader workers (cluster CPUs)
MODEL_NAME    = "yolo11s-pose"   # options: yolo11n-pose (fastest) / yolo11s-pose / yolo11m-pose

print(f"Model  : {MODEL_NAME}")
print(f"Epochs : {EPOCHS}")
print(f"ImgSz  : {IMGSZ}")
print(f"Batch  : {BATCH}")
print(f"Workers: {NUM_WORKERS}")

In [ ]:
# ── Verify labels, delete stale caches, then train ───────────────────────────
# Labels from PoseAutoLabeler have 35 columns:
#   1 (class) + 4 (bbox) + 10 keypoints × 3 (x, y, visibility) = 35

_lbl_dir = DATASET_DIR / "train" / "labels"
_sample  = next(_lbl_dir.glob("*.txt"), None)
assert _sample, f"No label files in {_lbl_dir}"
_first = [l for l in _sample.read_text().splitlines() if l.strip()][0]
_ncols = len(_first.split())
print(f"Label cols: {_ncols}  ({_sample.name})")
print(f"  {_first[:140]}")

_expected = 35   # 1 + 4 + 10×3
if _ncols == _expected:
    print(f"✓ Labels are correct ({_ncols} cols = 10 keypoints with visibility)")
else:
    print(f"⚠  Expected {_expected} cols, got {_ncols}")
    print("   Re-run PoseAutoLabeler to regenerate labels, or check export format.")
    raise RuntimeError(f"Label column mismatch: got {_ncols}, expected {_expected}")

# Delete stale YOLO label caches
_deleted = list(DATASET_DIR.rglob("*.cache"))
for _p in _deleted:
    _p.unlink()
print(f"Deleted {len(_deleted)} stale cache file(s)")

# Final yaml sanity-check
import yaml as _yaml
with open(patched_yaml) as _f:
    _cfg = _yaml.safe_load(_f)
assert _cfg.get("kpt_shape") == [10, 3], f"kpt_shape wrong: {_cfg.get('kpt_shape')}"
assert _cfg.get("nc") == 2,              f"nc wrong: {_cfg.get('nc')}"
print(f"patched_yaml OK: kpt_shape={_cfg['kpt_shape']}  nc={_cfg['nc']}  "
      f"names={_cfg['names']}")

# ── Train ─────────────────────────────────────────────────────────────────────
model   = YOLO(f"{MODEL_NAME}.pt")
results = model.train(
    data        = str(patched_yaml),
    epochs      = EPOCHS,
    imgsz       = IMGSZ,
    batch       = BATCH,
    workers     = NUM_WORKERS,
    device      = DEVICE,
    lr0         = LR0,
    lrf         = LRF,
    freeze      = FREEZE_LAYERS,
    patience    = PATIENCE,
    project     = str(OUTPUT_DIR),
    name        = "port_pose",
    exist_ok    = True,
    hsv_h       = 0.015,
    hsv_s       = 0.3,
    hsv_v       = 0.3,
    degrees     = 5.0,
    translate   = 0.05,
    scale       = 0.3,
    fliplr      = 0.0,   # NIC card has fixed orientation — no horizontal flip
    flipud      = 0.0,
    mosaic      = 0.5,
    copy_paste  = 0.0,
    verbose     = True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nBest model: {best_pt}")

## Evaluate on validation set

In [ ]:
# Load best weights and run validation
best_model = YOLO(str(best_pt))
val_metrics = best_model.val(data=str(patched_yaml), device=DEVICE, verbose=True)
print(f"\nmAP50-95 (pose): {val_metrics.pose.map:.4f}")
print(f"mAP50    (pose): {val_metrics.pose.map50:.4f}")

In [ ]:
# Visualise predictions on validation images
val_img_dir = DATASET_DIR / "valid" / "images"
val_imgs    = sorted(val_img_dir.glob("*.*"))[:8]
val_lbl_dir = DATASET_DIR / "valid" / "labels"

fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for ax, img_path in zip(axes.flatten(), val_imgs):
    img_bgr = cv2.imread(str(img_path))
    H, W = img_bgr.shape[:2]

    # Ground truth
    lbl_path = val_lbl_dir / (img_path.stem + ".txt")
    if lbl_path.exists():
        for cls_id, bbox, kps in parse_yolo_pose_label(lbl_path, W, H):
            x1,y1,x2,y2 = [int(v) for v in bbox]
            cv2.rectangle(img_bgr,(x1,y1),(x2,y2),(0,220,0),2)  # green = GT bbox
            for i,(u,v,vis) in enumerate(kps):
                if vis > 0:
                    cv2.circle(img_bgr,(int(u),int(v)),5,(0,220,0),-1)

    # Prediction
    preds = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    if preds.keypoints is not None and len(preds.keypoints.xy) > 0:
        boxes = preds.boxes
        for bi in range(len(boxes)):
            bx1,by1,bx2,by2 = [int(v) for v in boxes.xyxy[bi].cpu().numpy()]
            conf  = float(boxes.conf[bi])
            cls_i = int(boxes.cls[bi])
            name  = class_names.get(cls_i, str(cls_i)) if isinstance(class_names, dict) \
                    else (class_names[cls_i] if cls_i < len(class_names) else str(cls_i))
            cv2.rectangle(img_bgr,(bx1,by1),(bx2,by2),(255,100,0),2)  # orange = pred
            cv2.putText(img_bgr,f"{name} {conf:.2f}",(bx1,by1-5),
                        cv2.FONT_HERSHEY_SIMPLEX,0.4,(255,100,0),1)
            kp_xy = preds.keypoints.xy[bi].cpu().numpy()  # (5,2)
            for i,(u,v) in enumerate(kp_xy):
                cv2.circle(img_bgr,(int(u),int(v)),5,(255,100,0),-1)

    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); ax.axis("off")

legend = [mpatches.Patch(color="green",  label="GT"),
          mpatches.Patch(color="orange", label="Pred")]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("Val predictions (green=GT bbox+kp, orange=pred)", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "val_predictions.png", dpi=100); plt.show()

## PnP pose evaluation on val set

In [ ]:
# 3D keypoints in port-local frame (entrance_link), meters.
# sfp_nic: one detection gives kp0-4 for port_0 and kp5-9 for port_1.
# Run solvePnP separately for each port using KP3D_SFP.
KP3D_SFP = np.array([
    [ 0.000,  0.000, 0.0],   # 0  center
    [-0.006, -0.004, 0.0],   # 1  top-left
    [ 0.006, -0.004, 0.0],   # 2  top-right
    [ 0.006,  0.004, 0.0],   # 3  bottom-right
    [-0.006,  0.004, 0.0],   # 4  bottom-left
], dtype=np.float64)

KP3D_SC = np.array([
    [ 0.000,  0.000,  0.0],  # 0  center
    [ 0.0013, 0.000,  0.0],  # 1  right
    [ 0.000,  0.0013, 0.0],  # 2  bottom
    [-0.0013, 0.000,  0.0],  # 3  left
    [ 0.000, -0.0013, 0.0],  # 4  top
], dtype=np.float64)

def cls_name(cls_id):
    return class_names[cls_id] if cls_id < len(class_names) else str(cls_id)

# Load camera intrinsics
cam_info_candidates = [
    BASE_PATH / "camera_info.json",
    BASE_PATH.parent / "pose_data" / "camera_info.json",
    Path("../pose_data/camera_info.json"),
]
cam_info_path = next((p for p in cam_info_candidates if p.exists()), None)
if cam_info_path:
    with open(cam_info_path) as f:
        cam = json.load(f)
    K = np.array(cam["K"], dtype=np.float64).reshape(3, 3)
    D = np.zeros(5, dtype=np.float64)
    print(f"Camera K loaded from {cam_info_path}")
    print(K)
else:
    print("⚠  camera_info.json not found — copy pose_data/camera_info.json here")
    K = None

In [ ]:
if K is None:
    print("Skipping PnP eval — no camera_info.json")
else:
    # Scale K from calibration resolution to inference resolution
    orig_w = cam.get("width",  1152)
    orig_h = cam.get("height", 1024)

    t_errors, r_errors = [], []
    val_imgs_all = sorted(val_img_dir.glob("*.*"))

    for img_path in val_imgs_all:
        img = cv2.imread(str(img_path))
        H, W = img.shape[:2]
        lbl_path = val_lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue

        K_img = K.copy()
        K_img[0, 0] *= W / orig_w;  K_img[0, 2] *= W / orig_w
        K_img[1, 1] *= H / orig_h;  K_img[1, 2] *= H / orig_h

        gt_instances = parse_yolo_pose_label(lbl_path, W, H)
        preds = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
        if preds.keypoints is None or len(preds.keypoints.xy) == 0:
            continue

        for bi in range(len(preds.boxes)):
            cls_i = int(preds.boxes.cls[bi])
            name  = cls_name(cls_i)
            pred_kp_all = preds.keypoints.xy[bi].cpu().numpy()   # (10, 2)

            # Find matching GT instance by class
            gt_match = [(c, b, k) for c, b, k in gt_instances if c == cls_i]
            if not gt_match:
                continue
            _, _, gt_kps = gt_match[0]   # list of (u,v,vis) × 10

            # Determine which port slices to evaluate
            if name == "sfp_nic":
                port_slices = [(0, slice(0, 5)), (1, slice(5, 10))]
                kp3d = KP3D_SFP
            else:   # sc_port
                port_slices = [(0, slice(0, 5))]
                kp3d = KP3D_SC

            for port_idx, sl in port_slices:
                pred_kp = pred_kp_all[sl].astype(np.float64)       # (5, 2)
                gt_kp   = np.array([[u, v] for u, v, vis in gt_kps[sl]
                                    if vis > 0], dtype=np.float64)
                if len(gt_kp) < 4:
                    continue   # not enough visible GT kps for this port

                ok_p, rv_p, tv_p = cv2.solvePnP(
                    kp3d, pred_kp, K_img, D, flags=cv2.SOLVEPNP_ITERATIVE)
                ok_t, rv_t, tv_t = cv2.solvePnP(
                    kp3d, gt_kp[:5] if len(gt_kp) >= 5 else gt_kp,
                    K_img, D, flags=cv2.SOLVEPNP_ITERATIVE)
                if not (ok_p and ok_t):
                    continue

                t_errors.append(float(np.linalg.norm(tv_p - tv_t)) * 1000.0)
                R_p, _ = cv2.Rodrigues(rv_p)
                R_t, _ = cv2.Rodrigues(rv_t)
                tr = np.clip((np.trace(R_p @ R_t.T) - 1.0) / 2.0, -1.0, 1.0)
                r_errors.append(math.degrees(math.acos(tr)))

    if t_errors:
        print(f"PnP on {len(t_errors)} matched port instances:")
        print(f"  Translation: mean={np.mean(t_errors):.1f}mm  "
              f"median={np.median(t_errors):.1f}mm  "
              f"p90={np.percentile(t_errors,90):.1f}mm")
        print(f"  Rotation:    mean={np.mean(r_errors):.2f}°  "
              f"median={np.median(r_errors):.2f}°  "
              f"p90={np.percentile(r_errors,90):.2f}°")
        print()
        print("Target: translation < 5 mm  AND  rotation < 10°")
    else:
        print("No matched instances — check class names and detection confidence")

## Export and package outputs

In [ ]:
# Export best.pt to ONNX for CPU inference on the robot (optional)
best_model.export(format="onnx", imgsz=IMGSZ, opset=12)
onnx_path = best_pt.with_suffix(".onnx")
print(f"ONNX model: {onnx_path}")

# Copy key files to OUTPUT_DIR for easy access
shutil.copy(best_pt, OUTPUT_DIR / "best.pt")
if onnx_path.exists():
    shutil.copy(onnx_path, OUTPUT_DIR / "best.onnx")
shutil.copy(patched_yaml, OUTPUT_DIR / "data.yaml")

print(f"\nOutput files:")
for p in sorted(OUTPUT_DIR.glob("*.*")):
    print(f"  {p.name:40s}  {p.stat().st_size/1024/1024:.1f} MB")

In [ ]:
# Package outputs into a zip for easy download from JupyterHub
import zipfile as _zf

zip_path = BASE_PATH / "yolo_output.zip"
with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
    for ext in ("*.pt", "*.onnx", "*.yaml", "*.png"):
        for src in OUTPUT_DIR.glob(ext):
            zf.write(src, src.name)

print(f"Download this file from JupyterHub: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1024 / 1024:.1f} MB")
print("\nContents:")
for p in sorted(OUTPUT_DIR.glob("*.*")):
    print(f"  {p.name:40s}  {p.stat().st_size/1024/1024:.1f} MB")

## After training — files to download

Download `yolo_output.zip` from JupyterHub (right-click → Download in the file browser).

Extract and place under `~/ws_aic/src/aic/my_policy_node/pose_model/`:

```
pose_model/
  best.pt        ← YOLOv11-pose weights
  best.onnx      ← ONNX export for CPU inference (optional)
  data.yaml      ← class names needed by PoseEstimator.py
  camera_info.json
```

## Loading in PoseEstimator.py

```python
from ultralytics import YOLO
model = YOLO("pose_model/best.pt")

# At inference:
results = model.predict(img_rgb, conf=0.4, verbose=False)[0]
if results.keypoints and len(results.keypoints.xy) > 0:
    cls_id  = int(results.boxes.cls[0])
    kp_2d   = results.keypoints.xy[0].cpu().numpy()   # (5, 2) pixels
    # → solvePnP with KP3D[class_name] → port pose TF frame
```

## Confidence threshold

Start with `conf=0.4`.  If the port is occasionally missed (robot is far away), lower to 0.25.
If wrong detections appear, raise to 0.5.